In [142]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [143]:
data = pd.read_csv('data/dataset_logical_filter.csv')
df = pd.DataFrame(data)
df

,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0
...,...,...,...,...,...,...,...,...
292,2016,9.50,11.60,33988,Diesel,Dealer,Manual,0
293,2015,4.00,5.90,60000,Petrol,Dealer,Manual,0
294,2009,3.35,11.00,87934,Petrol,Dealer,Manual,0
295,2017,11.50,12.50,9000,Diesel,Dealer,Manual,0


In [144]:
print(df['Fuel_Type'].value_counts())

Fuel_Type
Petrol    239
Diesel     58
Name: count, dtype: int64


In [145]:
df

,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0
...,...,...,...,...,...,...,...,...
292,2016,9.50,11.60,33988,Diesel,Dealer,Manual,0
293,2015,4.00,5.90,60000,Petrol,Dealer,Manual,0
294,2009,3.35,11.00,87934,Petrol,Dealer,Manual,0
295,2017,11.50,12.50,9000,Diesel,Dealer,Manual,0


In [146]:
df['Fuel_Type'] = df['Fuel_Type'].map({'Petrol': 0, 'Diesel': 1})
df['Seller_Type'] = df['Seller_Type'].map({'Dealer': 0, 'Individual': 1})
df['Transmission'] = df['Transmission'].map({'Manual': 0, 'Automatic': 1})

In [147]:
df['Car_Age'] = 2019 - df['Year']
df['Age_Squared'] = df['Car_Age'] ** 2
df['Kms_Squared'] = df['Kms_Driven'] ** 2
df['Age_Log'] = np.log1p(df['Car_Age'])
df

,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age,Age_Squared,Kms_Squared,Age_Log
0,2014,3.35,5.59,27000,0,0,0,0,5,25,729000000,1.791759
1,2013,4.75,9.54,43000,1,0,0,0,6,36,1849000000,1.945910
2,2017,7.25,9.85,6900,0,0,0,0,2,4,47610000,1.098612
3,2011,2.85,4.15,5200,0,0,0,0,8,64,27040000,2.197225
4,2014,4.60,6.87,42450,1,0,0,0,5,25,1802002500,1.791759
...,...,...,...,...,...,...,...,...,...,...,...,...
292,2016,9.50,11.60,33988,1,0,0,0,3,9,1155184144,1.386294
293,2015,4.00,5.90,60000,0,0,0,0,4,16,3600000000,1.609438
294,2009,3.35,11.00,87934,0,0,0,0,10,100,7732388356,2.397895
295,2017,11.50,12.50,9000,1,0,0,0,2,4,81000000,1.098612


In [148]:
print(df.isnull().sum())

Year             0
Selling_Price    0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner            0
Car_Age          0
Age_Squared      0
Kms_Squared      0
Age_Log          0
dtype: int64


In [149]:
X_raw = df.drop(['Selling_Price', 'Year', 'Age_Squared'], axis=1, errors='ignore')
y = df['Selling_Price']

In [150]:
scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
X_scaled_array = scaler.fit_transform(X_raw)
X = pd.DataFrame(X_scaled_array, columns=X_raw.columns)


In [151]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = LinearRegression()
r2_scores = []
mae_scores = []
mse_scores = []
fold_number = 1
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = r2_score(y_test, y_pred)
    r2_scores.append(score)
    print(f"{fold_number} | {score:.4f}")
    fold_number += 1
print(np.mean(r2_scores))

1 | 0.8982
2 | 0.8660
3 | 0.8680
4 | 0.8924
5 | 0.8627
0.8774394706995393


In [152]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [153]:
final_model = model.fit(X_train, y_train)
y_pred_final = model.predict(X_test)

In [154]:
mae = mean_absolute_error(y_test, y_pred_final)
mse = mean_squared_error(y_test, y_pred_final)
r2 = r2_score(y_test, y_pred_final)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")

R2 Score: 0.9287477236333764
MAE: 1.0073738853462049
MSE: 1.562601092893053


In [155]:
df_final = df

In [156]:
df_final['Present_Price_Log'] = np.log1p(df_final['Present_Price'])
df_final['Kms_Driven_Log'] = np.log1p(df_final['Kms_Driven'])
X_raw = df_model.drop([
    'Selling_Price',
    'Year',
    'Age_Squared',
    'Kms_Squared',
    'Age_Log',
    'Present_Price',
    'Kms_Driven',
], axis=1, errors='ignore')

y_log = np.log(df['Selling_Price'])
scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
X_scaled_array = scaler.fit_transform(X_raw)
X = pd.DataFrame(X_scaled_array, columns=X_raw.columns)

In [157]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model_log_target = LinearRegression()

r2_scores, mae_scores = [], []
fold = 1
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_log.iloc[train_index], y_log.iloc[test_index]

    model_log_target.fit(X_train, y_train)
    y_pred_log = model_log_target.predict(X_test)


    y_test_real = np.exp(y_test)
    y_pred_real = np.exp(y_pred_log)

    r2 = r2_score(y_test_real, y_pred_real)
    mae = mean_absolute_error(y_test_real, y_pred_real)

    r2_scores.append(r2)
    mae_scores.append(mae)
    print(f"{fold} | R2: {r2:.4f} | MAE: {mae:.4f}")
    fold += 1

print(f"{np.mean(r2_scores):.4f}")

1 | R2: 0.9332 | MAE: 0.9023
2 | R2: 0.9449 | MAE: 0.5392
3 | R2: 0.9551 | MAE: 0.5067
4 | R2: 0.9586 | MAE: 0.5819
5 | R2: 0.9202 | MAE: 0.6333
0.9424


In [158]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

In [159]:
test_model = model.fit(X_train, y_train)
y_pred_test_log = test_model.predict(X_test)

In [160]:
y_test_real = np.exp(y_test)
y_pred_test_real = np.exp(y_pred_test_log)

In [161]:
mae_real = mean_absolute_error(y_test_real, y_pred_test_real)
mse_real = mean_squared_error(y_test_real, y_pred_test_real)
r2_real = r2_score(y_test_real, y_pred_test_real)

print(f"R2 Score (Real): {r2_real:.4f}")
print(f"MAE (Real): {mae_real:.4f}")
print(f"MSE (Real): {mse_real:.4f}")

R2 Score (Real): 0.9332
MAE (Real): 0.9023
MSE (Real): 2.1085
